In [1]:
# import required python modules
import os
import re
import unicodedata
import fitz
import pandas as pd

In [1]:
# set input and output paths
base_dir = r"D:\Egypt-Opportunity-Radar-Task3\data\raw\production"
in_path = os.path.join(base_dir, "task3_capmas_public_commodity_production_fy2022_23.pdf")
out_dir = os.path.join(base_dir, "cleaned")
os.makedirs(out_dir, exist_ok=True)

In [1]:
# open raw pdf document
doc = fitz.open(in_path)
commodities = []
page_summary = []

In [1]:
# parse pages and tables
for idx, page in enumerate(doc):
    text = unicodedata.normalize("NFKC", page.get_text("text"))
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    imgs = len(page.get_images())
    ptype = "picture only" if not lines and imgs > 0 else "single table page"
    page_summary.append({"page": idx + 1, "lines": len(lines), "images": imgs, "page_type": ptype})
    if idx + 1 >= 14:
        i = 0
        while i + 2 < len(lines):
            if lines[i].isdigit() and re.match(r"^\d+", lines[i + 1]) and not lines[i + 2].isdigit():
                m = re.match(r"^(\d+)\s*(.*)$", lines[i + 1])
                qty, unit = (int(m.group(1)), m.group(2).strip()) if m else (0, "")
                commodities.append({"page": idx + 1, "commodity_ar": lines[i + 2], "unit_ar": unit, "quantity": qty, "value_thousand_egp": int(lines[i])})
                i += 3
            else:
                i += 1

In [1]:
# build cleaned dataframes
df_comm = pd.DataFrame(commodities)
df_pages = pd.DataFrame(page_summary)

In [1]:
# save cleaned commodity files
csv_out = os.path.join(out_dir, "cleaned_task3_capmas_public_commodity_production_fy2022_23.csv")
df_comm.to_csv(csv_out, index=False, encoding="utf-8-sig")

In [1]:
xlsx_out = os.path.join(out_dir, "cleaned_task3_capmas_public_commodity_production_fy2022_23.xlsx")
with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
    df_comm.to_excel(writer, sheet_name="commodities_fy22_23", index=False)
    df_pages.to_excel(writer, sheet_name="page_summary", index=False)